# channel_no_conc Batch Workflow

批量入口 notebook：扫描一个目录下第一层的主配置 JSON，展开其中引用的全部模板，并逐个运行 `channel_no_conc` compare。


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_repo_root(start=None):
    cwd = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "examples" / "neuron_compare" / "channel_no_conc" / "workflows" / "workflow_api.py").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


REPO_ROOT = find_repo_root()
CHANNEL_NO_CONC_ROOT = REPO_ROOT / "examples" / "neuron_compare" / "channel_no_conc"
WORKFLOWS_ROOT = CHANNEL_NO_CONC_ROOT / "workflows"
if str(WORKFLOWS_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKFLOWS_ROOT))

import workflow_api
import brainstate
brainstate.environ.set(precision=64)


## Parameters

| 参数 | 含义 | 常见取值 |
|---|---|---|
| `config_dir` | 批量扫描目录；只扫描第一层 `*.json`，每个 config 会跑完自己声明的全部模板。 | `CHANNEL_NO_CONC_ROOT / "configs"` |
| `summary_dir` | 本次 batch run 的输出根目录。 | `None` 自动生成为 `results/batch_runs/YYMMDD_HHMMSS/`；或手动指定路径 |
| `plot_cases` | 是否在每个 config 的模板子目录里生成图产物。 | `True` / `False` |


In [ ]:
config_dir = CHANNEL_NO_CONC_ROOT / "configs" / "ma24_pc"  # 批量扫描目录；每个 config 会跑完自己声明的全部模板
summary_dir = None  # 本次 batch run 的输出根目录；None 时自动使用 YYMMDD_HHMMSS 命名
plot_cases = True # 是否在每个 config 的模板子目录里生成图产物


## Discover Configs And Prepare Batch Directory


In [ ]:
config_records = workflow_api.discover_batch_configs(config_dir)
batch_run_id = workflow_api.make_batch_run_id()
resolved_summary_dir = workflow_api.default_batch_run_output_dir(
    batch_run_id=batch_run_id,
    summary_dir=summary_dir,
)
configs_out_dir = resolved_summary_dir / "configs"
configs_out_dir.mkdir(parents=True, exist_ok=True)

preview_df = pd.DataFrame(
    [
        {
            "config_name": row["config_name"],
            "config_path": str(row["config_path"]),
            "n_templates": row["n_templates"],
            "template_names": ", ".join(row["template_names"]),
            "config_out_dir": str(configs_out_dir / row["config_name"]),
        }
        for row in config_records
    ]
)
print("batch_run_id:", batch_run_id)
print("resolved_summary_dir:", resolved_summary_dir)
display(preview_df)


## Run Each Config


In [4]:
config_run_infos = []
for record in config_records:
    config_out_dir = configs_out_dir / str(record["config_name"])
    run_info = workflow_api.run_notebook_config_workflow(
        config_path=record["config_path"],
        out_dir=config_out_dir,
        plot=plot_cases,
        expand_only=False,
        raise_on_failure=False,
    )
    config_run_infos.append(run_info)

batch_summary = workflow_api.write_batch_summary_artifacts(
    config_dir=config_dir,
    summary_dir=resolved_summary_dir,
    batch_run_id=batch_run_id,
    config_records=config_records,
    config_run_infos=config_run_infos,
    plot_cases=plot_cases,
)
print("batch_manifest:", batch_summary["manifest_path"])
print("batch_observable_summary_csv:", batch_summary["batch_observable_summary_path"])
print("batch_observable_summary_json:", batch_summary["batch_observable_summary_json_path"])
print("batch_failures_csv:", batch_summary["batch_failures_path"])


batch_manifest: /home/swl/braincell/examples/neuron_compare/channel_no_conc/results/batch_runs/260426_145159/batch_manifest.json
batch_observable_summary_csv: /home/swl/braincell/examples/neuron_compare/channel_no_conc/results/batch_runs/260426_145159/batch_observable_summary.csv
batch_observable_summary_json: /home/swl/braincell/examples/neuron_compare/channel_no_conc/results/batch_runs/260426_145159/batch_observable_summary.json
batch_failures_csv: /home/swl/braincell/examples/neuron_compare/channel_no_conc/results/batch_runs/260426_145159/batch_failures.csv


## Load Batch Summary Tables


In [5]:
runs_df = pd.read_csv(batch_summary["config_runs_path"]).sort_values(["batch_status", "config_name"]).reset_index(drop=True)
observables_df = pd.read_csv(batch_summary["batch_observable_summary_path"]).sort_values(["observable", "config_name"]).reset_index(drop=True)
failures_df = pd.read_csv(batch_summary["batch_failures_path"]).reset_index(drop=True)

display(observables_df)
display(failures_df)


,config_name,config_path,batch_status,observable,n_cases,mae_mean,rmse_mean,max_abs_max,rel_mae_pct_mean
0,hcn1_ma24_pc,/home/swl/braincell/examples/neuron_compare/ch...,ok,current.ix,15,7.422202e-17,8.717340e-17,4.413787e-16,9.509458e-11
1,kir2p3_ma24_pc,/home/swl/braincell/examples/neuron_compare/ch...,ok,current.ix,15,1.787318e-07,1.996241e-07,9.069120e-07,1.022864e-02
2,kv1p1_ma24_pc,/home/swl/braincell/examples/neuron_compare/ch...,ok,current.ix,15,7.147042e-16,9.473506e-16,1.184903e-14,2.928036e-10
3,kv3p4_ma24_pc,/home/swl/braincell/examples/neuron_compare/ch...,ok,current.ix,15,4.774356e-16,5.942080e-16,5.731526e-15,1.225956e-10
4,kv4p3_ma24_pc,/home/swl/braincell/examples/neuron_compare/ch...,ok,current.ix,15,8.471918e-08,1.073866e-07,2.237126e-06,4.556533e-02
5,kv4p3_ma24_pc,/home/swl/braincell/examples/neuron_compare/ch...,ok,gates.a,15,8.758545e-05,1.769163e-04,6.430725e-03,9.133701e-02
6,kv4p3_ma24_pc,/home/swl/braincell/examples/neuron_compare/ch...,ok,gates.b,15,2.073285e-05,5.948565e-05,3.933530e-03,1.216603e-02
7,kir2p3_ma24_pc,/home/swl/braincell/examples/neuron_compare/ch...,ok,gates.d,15,2.933922e-05,3.137721e-05,9.509572e-05,1.564502e-02
8,hcn1_ma24_pc,/home/swl/braincell/examples/neuron_compare/ch...,ok,gates.h,15,2.037846e-15,2.706581e-15,1.470005e-14,1.183341e-11
9,kv3p4_ma24_pc,/home/swl/braincell/examples/neuron_compare/ch...,ok,gates.h,15,4.123835e-15,5.684995e-15,1.170175e-13,4.148106e-13


,config_name,config_path,batch_status,template_name,template_path,out_dir,n_failed_cases,error_message


如果某个 config 需要继续做 template / case 级钻取，进入该 batch run 目录下的 `configs/<config_name>/`，再用 [workflow.ipynb](/home/swl/braincell/examples/neuron_compare/channel_no_conc/workflows/workflow.ipynb) 对单个 config 继续分析。
